# JARVIS-VLA leftover-mouth ladder

One checkpoint: `CraftJarvis/JarvisVLA-Qwen2-VL-7B`. Kernel **jarvis-vqa**. Same weights as `demo.ipynb`. Not MineStudio.

Vendor source (clone in `vendor/JarvisVLA`, gitignored; `jarvisvla/evaluate/agent_wrapper.py` `VLLM_AGENT.forward`). Facts, not guesses:

- Official user text is `{instruction}\nobservation: ` then the image (`create_message_vllm`: text first, then image). Recipe mode *injects* a canned `thought:` on the **user** side; the assistant is still expected to emit action tokens.
- There is **no** forced assistant prefix like Molmo's `<depth_output><action_output>`. `apply_chat_template(..., add_generation_prompt=True)` ends at `<|im_start|>assistant\n`. The reserved-special stream is a trained reflex, not a harness muzzle.
- Action tokens are `<|reserved_special_token_N|>` (mapped in `jarvisvla/inference/action_mapping.py`). `<think>` exists in `assets/special_token.json` but VLLM_AGENT never uses those tags.

The 2026-09-03 VQA run already used this official layout (`{question}\nobservation: \n` + image) and got `action_tokens` only. These five probes leave that railroad one step at a time. Raw decode first (specials kept); formatted / verdict after, with space between panes.

If `screenshot/` is empty: `source .venv/bin/activate && python download_screenshots.py`


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from IPython.display import display, Markdown
from PIL import Image
from qwen_mouth import JarvisQwenMouth, decode_keep_specials, mouth_verdict, append_log

WEIGHTS = ROOT / "weights" / "JarvisVLA-Qwen2-VL-7B"
SHOT_DIR = ROOT / "screenshot"
SHOTS = sorted(
    p for p in SHOT_DIR.glob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
print("weights", WEIGHTS.exists(), WEIGHTS)
print("screenshots", len(SHOTS), SHOT_DIR)
if not SHOTS:
    raise FileNotFoundError(
        "screenshot/ is empty. In the jarvis-vqa venv: python download_screenshots.py"
    )
frame = Image.open(SHOTS[0]).convert("RGB")
display(Markdown(f"**frame** `{SHOTS[0].name}` {frame.size}"))
display(frame)


## Load

Restart the kernel if you already have another 7B in VRAM. Reuse `mouth` if `demo.ipynb` already loaded it in this kernel.


In [ ]:
if "mouth" not in globals():
    mouth = JarvisQwenMouth(WEIGHTS)
print("loaded", mouth.weights)


## Official rails (control)

Same layout as `demo.ipynb` / VLLM_AGENT `normal`: `{instruction}\nobservation: \n` then image. A Minecraft instruction, not a VQA question — this is the product loop.


In [ ]:
import torch
from qwen_vl_utils import process_vision_info

SENTINEL = "[START ACTION]"
VENDOR = ROOT / "vendor" / "JarvisVLA"
if VENDOR.is_dir() and str(VENDOR) not in sys.path:
    sys.path.insert(0, str(VENDOR))


def show_raw(title: str, text: str) -> None:
    display(Markdown(f"### {title}"))
    print()
    print(text if text else "(empty)")
    print()
    print()
    print()


def official_user_text(instruction: str) -> str:
    # VLLM_AGENT.forward, history_num=0, instruction_type != recipe:
    #   prompt_input = private_instruction + "\nobservation: "
    # VQA harness used a trailing newline after the colon; keep it.
    return f"{instruction}\nobservation: \n"


def messages_official(instruction: str, image):
    return [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": official_user_text(instruction)},
                {"type": "image", "image": image},
            ],
        }
    ]


def messages_plain(text: str, image=None):
    content = [{"type": "text", "text": text}]
    if image is not None:
        content.append({"type": "image", "image": image})
    return [{"role": "user", "content": content}]


def generate(messages, max_new_tokens: int = 128):
    proc = mouth.processor
    text = proc.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    kwargs = dict(text=[text], padding=True, return_tensors="pt")
    if image_inputs:
        kwargs["images"] = image_inputs
    if video_inputs:
        kwargs["videos"] = video_inputs
    inputs = proc(**kwargs)
    inputs = inputs.to(mouth.model.device)
    with torch.inference_mode():
        output = mouth.model.generate(**inputs, max_new_tokens=max_new_tokens)
    gen = output[0, inputs["input_ids"].shape[1] :]
    return inputs, gen, text


def raw_decode(gen_ids) -> str:
    ids = gen_ids.tolist() if hasattr(gen_ids, "tolist") else list(gen_ids)
    return decode_keep_specials(mouth.processor.tokenizer, ids)


def sentinel_cut(gen_ids):
    tok = mouth.processor.tokenizer
    ids = gen_ids.tolist()
    for i in range(1, len(ids) + 1):
        if SENTINEL in tok.decode(ids[:i], skip_special_tokens=False):
            return i
    return None


def continue_after(inputs, kept_gen_ids, extra_text: str = "", max_new_tokens: int = 96):
    tok = mouth.processor.tokenizer
    pieces = [inputs["input_ids"][0], kept_gen_ids]
    if extra_text:
        extra = torch.tensor(
            tok(extra_text, add_special_tokens=False)["input_ids"],
            device=inputs["input_ids"].device,
            dtype=inputs["input_ids"].dtype,
        )
        pieces.append(extra)
    ids = torch.cat(pieces).unsqueeze(0)
    new_inputs = {
        k: v for k, v in inputs.items() if k not in ("input_ids", "attention_mask")
    }
    new_inputs["input_ids"] = ids
    new_inputs["attention_mask"] = torch.ones_like(ids)
    with torch.inference_mode():
        output = mouth.model.generate(**new_inputs, max_new_tokens=max_new_tokens)
    return output[0, ids.shape[1] :]


def try_action_decode(ids):
    try:
        from jarvisvla.inference.action_mapping import OneActionTokenizer
        tok = OneActionTokenizer(tokenizer_type="qwen2_vl")
        return tok.decode(ids.tolist() if hasattr(ids, "tolist") else list(ids))
    except Exception as e:
        return f"(no action decode: {type(e).__name__}: {e})"


def show_formatted(tag: str, gen_ids, raw: str):
    verdict = mouth_verdict(raw)
    display(Markdown(f"#### formatted — {tag}"))
    print("verdict:", verdict)
    print("action map:", try_action_decode(gen_ids))
    return verdict


def probe(name: str, messages, expect_sentinel: bool, max_new_tokens: int = 128, tack: str = ""):
    display(Markdown(f"## {name}"))
    inputs, gen, prompt_text = generate(messages, max_new_tokens=max_new_tokens)
    print("chat template (ends at assistant turn):")
    print(prompt_text)
    print()
    print()
    raw1 = raw_decode(gen)
    show_raw("RAW — before sentinel (free continuation)", raw1)
    v1 = show_formatted("before sentinel", gen, raw1)
    rec = {
        "sandbox": "jarvis_vqa",
        "turn": "mouth_inject_probe",
        "probe": name,
        "prompt": prompt_text,
        "raw": raw1,
        "reply": raw1,
        "verdict": v1,
    }
    if expect_sentinel:
        cut = sentinel_cut(gen)
        rec["sentinel_at"] = cut
        if cut is None:
            show_raw("RAW — after sentinel", f"(sentinel {SENTINEL!r} never emitted — no phase 2)")
        else:
            print(f"sentinel {SENTINEL!r} at token {cut}; continuing (no official trigger to tack)")
            print()
            print()
            gen2 = continue_after(inputs, gen[:cut], extra_text=tack)
            raw2 = raw_decode(gen2)
            show_raw("RAW — after sentinel (same generate continued)", raw2)
            v2 = show_formatted("after sentinel", gen2, raw2)
            rec["phase2_raw"] = raw2
            rec["phase2_verdict"] = v2
    append_log(rec)
    return rec


### Probe 1 — official `observation:` rails, muzzle already off

VLLM_AGENT `normal` user text + image. First token is the model's. If it opens with `<|reserved_special_token_*|>`, the mouth is a trained reflex — there was never a harness trigger to strip.


In [ ]:
out1 = probe(
    "probe 1: official observation: rails",
    messages_official("mine a tree", frame),
    expect_sentinel=False,
)


### Probe 2 — English first, then the sentinel

Still the official `{instruction}\nobservation: ` wrapper. Contract: one English sentence about the frame, then exactly `[START ACTION]`. JARVIS has no official assistant trigger to re-attach (unlike Molmo `<depth_output>`); after the sentinel the harness just continues generation. After-pane = whatever the policy does once it has already spoken.


In [ ]:
PROBE_2 = (
    "mine a tree. first write one short English sentence about what you see "
    "in the screenshot. then, when you are ready to act, output exactly "
    "[START ACTION] and nothing else after it"
)
out2 = probe(
    "probe 2: English first, then sentinel",
    messages_official(PROBE_2, frame),
    expect_sentinel=True,
)


### Probe 3 — ambiguous: clarify or go

Official rails still. Underspecified Minecraft ask. Sure → `[START ACTION]`; unsure → one clarifying question in English, no sentinel. A question here is the SIMA-2 behavior; a blind reserved-token dump means the policy commits regardless.


In [ ]:
PROBE_3 = (
    "mine the tree. careful: there may be more than one tree, or none. "
    "if you can tell which tree is meant, output exactly [START ACTION]. "
    "if you cannot tell, do not output [START ACTION]; instead ask one "
    "clarifying question in English and stop"
)
out3 = probe(
    "probe 3: ambiguous task, clarify or go",
    messages_official(PROBE_3, frame),
    expect_sentinel=True,
)


### Probe 4 — drop `observation:` (off the SFT layout)

No `observation:` suffix. Plain VQA on the same frame via `apply_chat_template`. This is the leftover-mouth test the 2026-09-03 run never actually ran — that run still used the official railroad.


In [ ]:
out4 = probe(
    "probe 4: VQA, no observation: suffix",
    messages_plain(
        "How many trees are in this screenshot, and what biome is this? "
        "Answer in plain English.",
        image=frame,
    ),
    expect_sentinel=False,
    max_new_tokens=96,
)


### Probe 5 — no image, hello world

Text-only chat on the same weights. Standard LLM smoke: repeat `hello there`. If even this is reserved specials, the mouth is gone at every distance from the training distribution.


In [ ]:
out5 = probe(
    "probe 5: no image, hello world",
    messages_plain("Please repeat back exactly these two words: hello there"),
    expect_sentinel=False,
    max_new_tokens=16,
)


## Reading the results

| Outcome | Meaning |
|---|---|
| Probe 1 is only reserved specials | Same as 2026-09-03. The `observation:` layout still elicits the policy. |
| 2–3: English then sentinel, phase 2 is actions | Talk-then-act on one checkpoint. Worth a note — JARVIS never had a forced trigger, so this would be a real leftover mouth. |
| 2–3: reserved tokens immediately, no English | Contract ignored. Prompt work on the official layout is over. |
| 3: clarifying question, no sentinel | The SIMA-2 ask. Even once is a big note. |
| 4–5 English, 1–3 reserved | Mouth alive only off the `observation:` railroad. A talking gate is a different net (or Path K). |
| 4–5 reserved / garbage | SFT ate the Qwen mouth. Same verdict as the first VQA run, now without the railroad as an excuse. |

Everything logs to `logs/*.jsonl` (`turn`: `mouth_inject_probe`).
